In [1]:
import pandas as pd
import torch
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader

# 1. Load the data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Convert text labels to integers (Neural networks only understand math/numbers)
train_df['sentiment'] = train_df['sentiment'].map({'pos': 1, 'neg': 0})
test_df['sentiment'] = test_df['sentiment'].map({'pos': 1, 'neg': 0})

print(f"Training samples: {len(train_df)}")
print(f"GPU Available: {torch.cuda.is_available()}")

Training samples: 25000
GPU Available: False


In [2]:
# Initialize the tokenizer from Hugging Face
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# quick test to visualize
sample_text = "This movie was absolutely fantastic!"
tokens = tokenizer(
    sample_text,
    padding='max_length',
    max_length=15,       # Forcing a length of 15 for this example
    truncation=True,
    return_tensors="pt"  # Return PyTorch tensors
)

print("Original:", sample_text)
print("Token IDs:", tokens['input_ids'])
print("Attention Mask:", tokens['attention_mask'])

C:\Users\ssbds\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Original: This movie was absolutely fantastic!
Token IDs: tensor([[  101,  2023,  3185,  2001,  7078, 10392,   999,   102,     0,     0,
             0,     0,     0,     0,     0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]])


In [3]:
from torch.utils.data import Dataset, DataLoader
import torch

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Dataset objects
train_dataset = IMDBDataset(train_df['text'].to_numpy(), train_df['sentiment'].to_numpy(), tokenizer)
test_dataset = IMDBDataset(test_df['text'].to_numpy(), test_df['sentiment'].to_numpy(), tokenizer)

#  DataLoaders to batch the data
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("Datasets and DataLoaders are created and ready!")

Datasets and DataLoaders are created and ready!


In [4]:
import torch
from transformers import DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW # <-- Imported from PyTorch now!

# 1. Load the model and tell it we have 2 labels (pos/neg)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

# 2. Move the model to the GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# 3. Set up the Optimizer using PyTorch's native AdamW
optimizer = AdamW(model.parameters(), lr=2e-5)

# 4. Set the number of epochs
epochs = 3

print(f"Model loaded and moved to {device}!")

C:\Users\ssbds\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded and moved to cpu!


In [5]:
import time

model.train()

for epoch in range(epochs):
    print(f"\n======== Epoch {epoch + 1} / {epochs} ========")
    total_train_loss = 0
    start_time = time.time()

    # Iterate through our DataLoader batch by batch
    for step, batch in enumerate(train_loader):

        # 1. Move the batch's tensors to the GPU
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        # Clear old gradients
        model.zero_grad()

        outputs = model(
            b_input_ids,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        # Loss calculation
        loss = outputs.loss
        total_train_loss += loss.item()

        # Backpropagation
        loss.backward()

        # 5. Update the weights based on the gradients
        optimizer.step()

        # Print an update every 500 batches
        if step % 500 == 0 and not step == 0:
            elapsed = round(time.time() - start_time, 2)
            print(f"  Batch {step:>5,}  of  {len(train_loader):>5,}.    Loss: {loss.item():.4f}    Time: {elapsed}s")

    # Calculate average loss for the whole epoch
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"\n  Average training loss: {avg_train_loss:.4f}")
    print(f"  Epoch took: {round(time.time() - start_time, 2)}s")


======== Epoch 1 / 3 ========
  Batch   500  of  1,563.    Loss: 0.3225    Time: 3394.36s
  Batch 1,000  of  1,563.    Loss: 0.2181    Time: 6253.84s
  Batch 1,500  of  1,563.    Loss: 0.3514    Time: 9125.77s

  Average training loss: 0.3480
  Epoch took: 9480.88s

======== Epoch 2 / 3 ========
  Batch   500  of  1,563.    Loss: 0.0127    Time: 3429.58s
  Batch 1,000  of  1,563.    Loss: 0.2396    Time: 5594.89s
  Batch 1,500  of  1,563.    Loss: 0.1857    Time: 7730.12s

  Average training loss: 0.2143
  Epoch took: 7994.15s

======== Epoch 3 / 3 ========
  Batch   500  of  1,563.    Loss: 0.1923    Time: 2173.76s
  Batch 1,000  of  1,563.    Loss: 0.0331    Time: 4363.09s
  Batch 1,500  of  1,563.    Loss: 0.1161    Time: 6558.26s

  Average training loss: 0.1132
  Epoch took: 6828.28s


In [6]:

import numpy as np
import time
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Put the model in evaluation mode
# This turns off training-specific behaviors like "Dropout" layers
model.eval()

# Variables to store our results
predictions = []
true_labels = []

print("Running Evaluation on the unseen Test Set...")
start_time = time.time()

# 2. Disable gradient calculation
# We aren't updating weights anymore, so this saves massive amounts of GPU memory and time
with torch.no_grad():
    for step, batch in enumerate(test_loader):

        # Move tensors to GPU
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        # Forward pass (get predictions)
        outputs = model(b_input_ids, attention_mask=b_input_mask)

        # 3. Extract the predictions
        logits = outputs.logits
        batch_predictions = torch.argmax(logits, dim=1).cpu().numpy()
        batch_labels = b_labels.cpu().numpy()

        # Store them in our lists
        predictions.extend(batch_predictions)
        true_labels.extend(batch_labels)

        if step % 500 == 0 and not step == 0:
            print(f"  Evaluated Batch {step:>5,} of {len(test_loader):>5,}")

# 4. Calculate Final Metrics
accuracy = accuracy_score(true_labels, predictions)
conf_matrix = confusion_matrix(true_labels, predictions)

print(f"\nEvaluation complete in {round(time.time() - start_time, 2)}s!\n")
print(f"=====================================")
print(f" Test Accuracy: {accuracy * 100:.2f}%")
print(f"=====================================\n")

print("Confusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(classification_report(true_labels, predictions, target_names=['Negative (0)', 'Positive (1)']))

Running Evaluation on the unseen Test Set...
  Evaluated Batch   500 of 1,563
  Evaluated Batch 1,000 of 1,563
  Evaluated Batch 1,500 of 1,563

Evaluation complete in 2004.63s!

 Test Accuracy: 86.48%

Confusion Matrix:
[[11406  1094]
 [ 2287 10213]]

Classification Report:
              precision    recall  f1-score   support

Negative (0)       0.83      0.91      0.87     12500
Positive (1)       0.90      0.82      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.87      0.86      0.86     25000
weighted avg       0.87      0.86      0.86     25000



In [7]:
def predict_sentiment(review_text):
    # 1. Put the model in evaluation mode
    model.eval()

    # 2. Tokenize the user's raw text string
    inputs = tokenizer(
        review_text,
        return_tensors="pt",
        truncation=True,
        padding='max_length',
        max_length=128
    )

    # Move the tensors to the GPU
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    # 3. Run the model without calculating gradients
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

        # Extract the highest probability score
        prediction = torch.argmax(outputs.logits, dim=1).item()
    if prediction == 1:
        return "Positive"
    else:
        return "Negative"

test_review_1 = "This was the absolute best movie I have ever seen. The acting was incredible!"
test_review_2 = "I fell asleep halfway through. What a complete waste of time and money."
test_review_3 = "It started out a bit slow, but the ending totally made up for it."

print(f"Review 1: {predict_sentiment(test_review_1)}")
print(f"Review 2: {predict_sentiment(test_review_2)}")
print(f"Review 3: {predict_sentiment(test_review_3)}")

Review 1: Positive
Review 2: Negative
Review 3: Positive
